In [9]:
from __future__ import annotations

import html
import logging
import time
import warnings
from pathlib import Path

import camelot
import pdfplumber

warnings.filterwarnings("ignore")
for _name in ("camelot", "pdfminer", "pdfminer.pdfpage", "PIL", "pypdf"):
    logging.getLogger(_name).setLevel(logging.ERROR)

ROOT = Path(".").resolve()
SAMPLES_DIR = ROOT / "парсинг pdf данные"
OUTPUT_DIR = ROOT / "парсинг html camelot"

CAMELOT_FLAVOR = "lattice"  # при пустом результате на странице пробуем stream

DOCUMENT_CSS = """
body { font-family: "Times New Roman", Times, serif; font-size: 11pt; line-height: 1.35; margin: 2em; }
table { border-collapse: collapse; margin: 0.6em 0 1em; width: auto; max-width: 100%; }
th, td { border: 1px solid #888; padding: 4px 10px; vertical-align: middle; text-align: left; }
td.num, th.num { text-align: right; }
h1, h2, h3, p { margin: 0.25em 0 0.5em; font-weight: inherit; font-size: inherit; }
.doc-section { margin-bottom: 0.75em; }
"""


def wrap_html_document(body: str, title: str = "Document") -> str:
    return (
        "<!DOCTYPE html>\n<html lang=\"ru\">\n<head>\n"
        f"<meta charset=\"utf-8\"><title>{html.escape(title)}</title>\n"
        f"<style>{DOCUMENT_CSS}</style>\n</head>\n<body>\n{body}\n</body>\n</html>"
    )


def cell_to_html(value) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""
    return html.escape(text).replace("\n", "<br>")


def raw_table_to_html(rows: list[list]) -> str:
    """Сырая сетка → <table> без классификаций/span."""
    if not rows:
        return ""
    n_cols = max((len(r) for r in rows), default=0)
    lines = ["<table>"]
    for row in rows:
        padded = list(row) + [None] * (n_cols - len(row))
        lines.append("<tr>")
        for cell in padded:
            lines.append(f"<td>{cell_to_html(cell)}</td>")
        lines.append("</tr>")
    lines.append("</table>")
    return "\n".join(lines)


def _camelot_read(pdf_path: Path, page_num: int, flavor: str):
    try:
        return camelot.read_pdf(
            str(pdf_path),
            pages=str(page_num),
            flavor=flavor,
            suppress_stdout=True,
        )
    except Exception:
        return None


def _camelot_bbox_to_plumber(bbox, page_height: float) -> tuple[float, float, float, float]:
    """Camelot PDF-coords (origin bottom-left) → pdfplumber (origin top-left)."""
    x0, y0, x1, y1 = bbox
    return (float(x0), float(page_height - y1), float(x1), float(page_height - y0))


def _point_in_bbox(x: float, y: float, bbox, pad: float = 2.0) -> bool:
    x0, top, x1, bottom = bbox
    return (x0 - pad) <= x <= (x1 + pad) and (top - pad) <= y <= (bottom + pad)


def _words_outside_tables(page, table_bboxes: list[tuple]) -> list[dict]:
    words = page.extract_words(use_text_flow=True, keep_blank_chars=False) or []
    if not table_bboxes:
        return words
    out: list[dict] = []
    for w in words:
        cx = (w["x0"] + w["x1"]) / 2.0
        cy = (w["top"] + w["bottom"]) / 2.0
        if any(_point_in_bbox(cx, cy, b) for b in table_bboxes):
            continue
        out.append(w)
    return out


def _words_to_paragraph_html(words: list[dict], y_tol: float = 3.0) -> str:
    if not words:
        return ""
    ordered = sorted(words, key=lambda w: (round(w["top"], 1), w["x0"]))
    lines: list[list[str]] = []
    cur_top: float | None = None
    cur: list[str] = []
    for w in ordered:
        t = w["top"]
        text = (w.get("text") or "").strip()
        if not text:
            continue
        if cur_top is None or abs(t - cur_top) <= y_tol:
            cur.append(text)
            if cur_top is None:
                cur_top = t
        else:
            if cur:
                lines.append(cur)
            cur = [text]
            cur_top = t
    if cur:
        lines.append(cur)
    if not lines:
        return ""
    body = "<br>\n".join(html.escape(" ".join(line)) for line in lines)
    return f"<p>{body}</p>"


def _cluster_words_into_blocks(
    words: list[dict], gap: float = 14.0
) -> list[tuple[float, list[dict]]]:
    if not words:
        return []
    ordered = sorted(words, key=lambda w: (w["top"], w["x0"]))
    blocks: list[list[dict]] = []
    cur: list[dict] = []
    prev_bottom: float | None = None
    for w in ordered:
        if prev_bottom is not None and (w["top"] - prev_bottom) > gap and cur:
            blocks.append(cur)
            cur = []
        cur.append(w)
        prev_bottom = max(prev_bottom or w["bottom"], w["bottom"])
    if cur:
        blocks.append(cur)
    return [(min(w["top"] for w in b), b) for b in blocks]


def build_page_body_camelot(pdf_path: Path, page_num: int, page) -> str:
    """Весь документ страницы: таблицы Camelot + текст вне них, сверху вниз."""
    result = _camelot_read(pdf_path, page_num, CAMELOT_FLAVOR)
    if result is None or result.n == 0:
        result = _camelot_read(pdf_path, page_num, "stream")

    table_sections: list[tuple[float, str]] = []
    table_bboxes: list[tuple] = []
    if result is not None:
        for table in result:
            rows = [list(row) for row in table.data]
            html_table = raw_table_to_html(rows)
            if not html_table:
                continue
            raw_bbox = getattr(table, "_bbox", None)
            if raw_bbox is None:
                report = getattr(table, "parsing_report", None) or {}
                raw_bbox = report.get("bbox") if isinstance(report, dict) else None
            if raw_bbox is not None:
                plumber_bbox = _camelot_bbox_to_plumber(raw_bbox, page.height)
            else:
                # без bbox не вычитаем зону — лучше риск дубля, чем потерять prose
                plumber_bbox = (0.0, 0.0, 0.0, 0.0)
            table_bboxes.append(plumber_bbox)
            table_sections.append(
                (plumber_bbox[1], f'<div class="doc-section">\n{html_table}\n</div>')
            )

    # пустой bbox (0,0,0,0) не должен маскировать весь текст
    table_bboxes = [b for b in table_bboxes if (b[2] - b[0]) > 1 and (b[3] - b[1]) > 1]

    prose_words = _words_outside_tables(page, table_bboxes)
    prose_sections: list[tuple[float, str]] = []
    for top, block_words in _cluster_words_into_blocks(prose_words):
        p_html = _words_to_paragraph_html(block_words)
        if p_html:
            prose_sections.append((top, f'<div class="doc-section">{p_html}</div>'))

    if not table_sections and not prose_sections:
        text = (page.extract_text() or "").strip()
        if text:
            return (
                '<div class="doc-section"><p>'
                + html.escape(text).replace("\n", "<br>")
                + "</p></div>"
            )
        return ""

    merged = sorted(table_sections + prose_sections, key=lambda x: x[0])
    return "\n".join(html_part for _, html_part in merged)


def pdf_to_html_camelot(pdf_path: str | Path) -> str:
    pdf_path = Path(pdf_path)
    sections: list[str] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for pnum, page in enumerate(pdf.pages, start=1):
            body = build_page_body_camelot(pdf_path, pnum, page)
            sections.append(f'<section class="page" data-page="{pnum}">\n{body}\n</section>')
    return wrap_html_document("\n".join(sections), title=pdf_path.stem)


## Прогон всех PDF из `парсинг pdf данные`

Результат → `парсинг html camelot/`.
Запускай ячейку ниже, когда будешь готов к экспорту (сам агент прогон не делал).

In [10]:
def export_parsing_camelot(
    samples_dir: str | Path = SAMPLES_DIR,
    output_dir: str | Path = OUTPUT_DIR,
) -> dict:
    import pdfplumber

    samples_dir = Path(samples_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pdfs = sorted(samples_dir.glob("*.pdf"))
    exported: list[str] = []
    total_pages = 0
    t0 = time.perf_counter()

    for pdf_path in pdfs:
        file_t0 = time.perf_counter()
        doc_html = pdf_to_html_camelot(pdf_path)
        with pdfplumber.open(str(pdf_path)) as pdf:
            n_pages = len(pdf.pages)
        total_pages += n_pages
        out_path = output_dir / f"{pdf_path.stem}.html"
        out_path.write_text(doc_html, encoding="utf-8")
        exported.append(out_path.name)
        print(
            f"{pdf_path.name} -> {out_path.name} "
            f"({n_pages} стр., {time.perf_counter() - file_t0:.2f} с)",
            flush=True,
        )

    elapsed = time.perf_counter() - t0
    return {
        "pdf_count": len(pdfs),
        "total_pages": total_pages,
        "total_seconds": round(elapsed, 4),
        "exported_files": exported,
        "output_dir": str(output_dir),
    }


# Раскомментируй / запусти для полного прогона:
# stats = export_parsing_camelot()
# print(stats)


In [11]:
stats = export_parsing_camelot()
print(stats)

1655388160-14.pdf -> 1655388160-14.html (1 стр., 6.04 с)
1655388160-26.pdf -> 1655388160-26.html (1 стр., 4.16 с)
1655388160-6.pdf -> 1655388160-6.html (1 стр., 2.92 с)
2508064833-12.pdf -> 2508064833-12.html (1 стр., 4.89 с)
2508064833-15.pdf -> 2508064833-15.html (1 стр., 6.87 с)
2508064833-17.pdf -> 2508064833-17.html (1 стр., 2.21 с)
2508064833-31.pdf -> 2508064833-31.html (1 стр., 2.38 с)
2703000015-111.pdf -> 2703000015-111.html (1 стр., 4.32 с)
2703000015-20.pdf -> 2703000015-20.html (1 стр., 2.35 с)
2703000015-24.pdf -> 2703000015-24.html (1 стр., 3.28 с)
2703000015-39.pdf -> 2703000015-39.html (1 стр., 1.51 с)
2703000015-46.pdf -> 2703000015-46.html (1 стр., 2.02 с)
2703000015-6.pdf -> 2703000015-6.html (1 стр., 2.35 с)
2703000015-9.pdf -> 2703000015-9.html (1 стр., 3.14 с)
2703000015-96.pdf -> 2703000015-96.html (1 стр., 2.02 с)
chet_f.pdf -> chet_f.html (1 стр., 2.03 с)
chet_f_2.pdf -> chet_f_2.html (1 стр., 2.64 с)
ks3.pdf -> ks3.html (1 стр., 1.93 с)
obrazec-schyota.pdf ->